# VRAGANT AI — Training Model Lokal (TF-IDF)

Notebook ini melatih model TF-IDF dari dataset parfum lokal Indonesia
dan mengekspor hasilnya ke file `.pkl` / `.npy` yang langsung bisa
dipakai oleh aplikasi Streamlit VRAGANT tanpa internet.

**Kenapa TF-IDF, bukan Random Forest?**
Dataset ini punya 169 kategori unik dari 418 produk (rata-rata ~2.5
produk per kategori). Random Forest butuh puluhan contoh per kelas
untuk belajar dengan baik -- dengan distribusi seperti ini hasilnya
tidak akan akurat. TF-IDF + cosine similarity lebih cocok karena
bekerja dari KONTEN teks secara langsung, bukan dari label kategori.

**Output yang dihasilkan:**
- `models/tfidf_vectorizer.pkl` — model TF-IDF yang sudah dilatih
- `models/tfidf_matrix.npy` — matrix vektor semua produk (418 x 3000)
- `models/produk_ids.npy` — mapping baris ke id_produk


In [ ]:
# ── LANGKAH 1: Install dependencies ────────────────────────────────────
!pip install scikit-learn pandas numpy -q

In [ ]:
# ── LANGKAH 2: Upload dataset ──────────────────────────────────────────
# Jalankan sel ini untuk upload file data_parfum.csv dari komputer kamu
from google.colab import files
print('Pilih file data_parfum.csv dari folder project kamu')
uploaded = files.upload()

In [ ]:
# ── LANGKAH 3: Load dan bersihkan dataset ─────────────────────────────
import pandas as pd
import numpy as np

df = pd.read_csv('data_parfum.csv', sep=';')
df = df.dropna(subset=['id_produk'])
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df['gender'] = df['gender'].str.strip().str.lower()
df['waktu']  = df['waktu'].str.strip().str.lower()
df['id_produk'] = df['id_produk'].astype(int)
df['harga'] = pd.to_numeric(df['harga'], errors='coerce')

print(f'Dataset dimuat: {len(df)} produk, {df.shape[1]} kolom')
print(f'Distribusi gender: {df["gender"].value_counts().to_dict()}')
print(f'Jumlah kategori unik: {df["kategori"].nunique()}')
df.head(3)

In [ ]:
# ── LANGKAH 4: Eksplorasi data ─────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['gender'].value_counts().plot(kind='bar', ax=axes[0], color=['#993C1D','#F0997B','#4A3B47'])
axes[0].set_title('Distribusi Gender')
axes[0].tick_params(axis='x', rotation=0)

df['waktu'].value_counts().plot(kind='bar', ax=axes[1], color=['#993C1D','#F0997B','#4A3B47'])
axes[1].set_title('Distribusi Waktu')
axes[1].tick_params(axis='x', rotation=0)

df['harga'].hist(ax=axes[2], bins=20, color='#993C1D', edgecolor='white')
axes[2].set_title('Distribusi Harga (Rp)')

plt.tight_layout()
plt.show()
print(f'Range harga: Rp{df["harga"].min():,.0f} - Rp{df["harga"].max():,.0f}')

In [ ]:
# ── LANGKAH 5: Siapkan fitur teks untuk TF-IDF ────────────────────────
# Gabungkan semua kolom teks yang relevan jadi 1 string per produk.
# Ini yang jadi 'representasi' tiap parfum dalam ruang TF-IDF.
#
# FIX (revisi): sebelumnya kolom 'aktivitas' dan 'waktu' TIDAK ikut
# digabung, padahal ScentDaily di app.py menyusun query dari kata-kata
# aktivitas/waktu (kuliah, nongkrong, siang, malam, outdoor, dst).
# Akibatnya vocabulary TF-IDF tidak mengenal kata-kata itu sama sekali
# (cek: 'kuliah', 'nongkrong', 'outdoor' tidak ada di vocabulary lama),
# jadi cosine similarity ScentDaily cuma kebetulan cocok di 1-2 kata,
# hasilnya itu-itu terus dan produk yang justru relevan tapi memakai
# kata lain (mis. Mykonos) tidak pernah terangkat.
# Sekarang kolom 'aktivitas' & 'waktu' (yang sudah ditulis manual per
# produk di data_parfum.csv) ikut dimasukkan sebagai fitur teks.
df['teks_fitur'] = (
    df['deskripsi_wangi'].fillna('') + ' ' +
    df['top_notes'].fillna('') + ' ' +
    df['middle_notes'].fillna('') + ' ' +
    df['base_notes'].fillna('') + ' ' +
    df['kategori'].fillna('') + ' ' +
    df['aktivitas'].fillna('') + ' ' +
    df['waktu'].fillna('')
)

print('Contoh teks fitur produk pertama:')
print(df['teks_fitur'].iloc[0])

In [ ]:
# ── LANGKAH 6: Training TF-IDF Vectorizer ─────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

# max_features=3000: ambil 3000 kata/frasa terpenting
# ngram_range=(1,2): tangkap frasa 2 kata juga (misal 'vanilla amber')
# min_df=1: masukkan semua kata yang muncul minimal 1 kali
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(df['teks_fitur'])

print(f'Matrix TF-IDF: {tfidf_matrix.shape}')
print(f'  - Baris: {tfidf_matrix.shape[0]} produk')
print(f'  - Kolom: {tfidf_matrix.shape[1]} fitur kata/frasa')
print(f'\n20 fitur/kata terpenting:')
print(vectorizer.get_feature_names_out()[:20].tolist())

In [ ]:
# ── LANGKAH 7: Uji model dengan query contoh ──────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

def cari_parfum(query, top_n=5, gender_filter=None):
    """Cari parfum berdasarkan query teks menggunakan TF-IDF lokal."""
    vec_query = vectorizer.transform([query])
    skor = cosine_similarity(vec_query, tfidf_matrix)[0]
    
    df_hasil = df.copy()
    df_hasil['skor'] = skor
    
    if gender_filter and gender_filter != 'unisex':
        df_hasil = df_hasil[df_hasil['gender'].isin([gender_filter, 'unisex'])]
    
    return df_hasil.sort_values('skor', ascending=False).head(top_n)[[
        'nama_parfum', 'brand', 'kategori', 'harga', 'skor'
    ]]

# Uji 1: query aktivitas
print('=== UJI 1: Query aktivitas kuliah pagi segar ===')
print(cari_parfum('kuliah pagi segar outdoor', gender_filter='male').to_string(index=False))

print('\n=== UJI 2: Query aroma woody oriental ===')
print(cari_parfum('kayu-kayuan hangat maskulin vanilla amber woody oriental').to_string(index=False))

print('\n=== UJI 3: Query acara formal malam ===')
print(cari_parfum('acara formal malam elegan mewah date night').to_string(index=False))

In [ ]:
# ── LANGKAH 8: Visualisasi TF-IDF (untuk laporan/presentasi) ──────────
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt

# Reduksi dimensi ke 2D untuk visualisasi
svd = TruncatedSVD(n_components=2, random_state=42)
matrix_2d = svd.fit_transform(tfidf_matrix)
matrix_2d = normalize(matrix_2d)

# Plot berdasarkan gender
warna_gender = {'male': '#4A3B47', 'female': '#F0997B', 'unisex': '#993C1D'}

plt.figure(figsize=(10, 7))
for g, warna in warna_gender.items():
    mask = df['gender'] == g
    plt.scatter(
        matrix_2d[mask, 0], matrix_2d[mask, 1],
        c=warna, label=g, alpha=0.6, s=30
    )
plt.title('Visualisasi Ruang TF-IDF (2D via SVD) — Warna = Gender')
plt.legend()
plt.tight_layout()
plt.savefig('tfidf_visualization.png', dpi=120)
plt.show()
print('Visualisasi tersimpan sebagai tfidf_visualization.png')

In [ ]:
# ── LANGKAH 9: Simpan model ke file ───────────────────────────────────
import pickle
import os

os.makedirs('models', exist_ok=True)

# Simpan vectorizer (berisi vocabulary + bobot IDF yang sudah dipelajari)
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# Simpan matrix sebagai numpy array (lebih compact untuk file)
np.save('models/tfidf_matrix.npy', tfidf_matrix.toarray())

# Simpan mapping id_produk (baris ke-N di matrix = produk dengan id ini)
np.save('models/produk_ids.npy', df['id_produk'].values)

print('Model berhasil disimpan:')
print(f'  models/tfidf_vectorizer.pkl ({os.path.getsize("models/tfidf_vectorizer.pkl") // 1024} KB)')
print(f'  models/tfidf_matrix.npy     ({os.path.getsize("models/tfidf_matrix.npy") // 1024} KB)')
print(f'  models/produk_ids.npy       ({os.path.getsize("models/produk_ids.npy") // 1024} KB)')

In [ ]:
# ── LANGKAH 10: Download semua file model ─────────────────────────────
# Setelah sel ini jalan, akan muncul prompt download untuk tiap file.
# Simpan ketiga file ke folder 'models/' di project VRAGANT kamu.
from google.colab import files

for nama_file in ['models/tfidf_vectorizer.pkl', 'models/tfidf_matrix.npy', 'models/produk_ids.npy']:
    files.download(nama_file)
    print(f'✓ {nama_file} siap didownload')

print('\nLangkah selanjutnya:')
print('1. Buat folder "models/" di dalam folder VRAGANT_NEW/')
print('2. Pindahkan ketiga file yang baru didownload ke sana')
print('3. Jalankan app: streamlit run app.py')